# GSPC-G2 retained sample: re-verify 4 signed cards

Pinned sample of 4 of the 31 signed GSPC-G2 cards (exact URLs and file sha256 below). For each card the notebook checks the file bytes against the pin, recomputes `id = sha256(canonical body)`, verifies Ed25519 against `did:web:csoai.org#board-attestation-1`, and shows n, accuracy and a Wilson 95% interval taken from the card body.

**Rules applied:** G2 is its own comparison group and is not mixed with board v2. The care card is included as a record but is `comparison_citable=false` (parse-confounded), so it is not a leader or a ranking. qwen3:8b safety is held and absent. Det cells are never citable. Measurement, not certification. No model is run.

## Licence

Code in this notebook: **MIT** (`SPDX-License-Identifier: MIT`), per the repo root `LICENSE` of CSOAI-ORG/councilof-ai at master 3139d7ad (Copyright (c) 2024-2026 COAI - Council of AIs).

Card data read by this notebook (the signed G2 card JSON): **CC-BY-4.0**.

In [1]:
# SPDX-License-Identifier: MIT
import json, hashlib, math, base64, urllib.request
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey

SAMPLE = [
    ('https://councilof.ai/interop/mill-cards-signed/signed-governan-91857b0eb93f.json',
     '91f5f6463fcc4739cd50d98532be6565930b44be3f1203b1afe63c9716eeeccd'),
    ('https://councilof.ai/interop/mill-cards-signed/signed-conforma-02f63fcd0876.json',
     '6b6d37fc56b0b6a233d6670f656f48133561ac005c000d4b07d3169dcaabf22e'),
    ('https://councilof.ai/interop/mill-cards-signed/signed-conforma-539a7bb52711.json',
     '43c30010a0f20a732298a9e88e19a973cdad889e37bdd51453a7e32d88df69e3'),
    ('https://councilof.ai/interop/mill-cards-signed/signed-care-e974e29df4cb.json',
     '0ca5e1f5cffc98e082bacc6a1d1168a553a09d15b44af8e6af397b35d57b72af'),
]
DID_URL = 'https://csoai.org/.well-known/did.json'
NOT_COMPARISON_CITABLE = {'care'}  # parse-confounded: record only, never leader/ranking

In [2]:
def get(u):
    req = urllib.request.Request(u, headers={'User-Agent': 'csoai-g2-notebook/1'})
    with urllib.request.urlopen(req) as r:
        return r.read()

def canon(body):
    # JS-canonical: recursively sorted keys, compact separators, UTF-8
    return json.dumps(body, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')

did = json.loads(get(DID_URL))
vm = {m['id']: m for m in did['verificationMethod']}
jwk = vm['did:web:csoai.org#board-attestation-1']['publicKeyJwk']
key = Ed25519PublicKey.from_public_bytes(base64.urlsafe_b64decode(jwk['x'] + '=='))
print('board-attestation-1 x =', jwk['x'])

board-attestation-1 x = k2fPWb6ctyu8l5at8FYgHsHFit_qoT-DssW3VNbCAXA


In [3]:
def wilson(k, n, z=1.959963984540054):
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

rows = []
for url, pin in SAMPLE:
    raw = get(url)
    card = json.loads(raw)
    b = card['body']
    pre = canon(b)
    rid = hashlib.sha256(pre).hexdigest()
    assert card['did'] == 'did:web:csoai.org#board-attestation-1'
    try:
        key.verify(bytes.fromhex(card['signature']), pre); sig = True
    except Exception:
        sig = False
    k = round(b['accuracy'] * b['n'])  # correct count implied by the card (accuracy is rounded to 4 dp)
    lo, hi = wilson(k, b['n'])
    rows.append(dict(file=url.rsplit('/',1)[1], model=b['model'].split('@')[0], axis=b['axis'],
        bytes_match_pin=hashlib.sha256(raw).hexdigest()==pin, id_ok=rid==card['id'], sig_ok=sig,
        n=b['n'], accuracy=b['accuracy'], k_implied=k, wilson95=f'[{lo:.4f}, {hi:.4f}]',
        parse_errors_excluded=b['compute_evidence']['parse_errors_excluded'],
        comparison_citable=b['axis'] not in NOT_COMPARISON_CITABLE))

for r in rows:
    print(json.dumps(r))

{"file": "signed-governan-91857b0eb93f.json", "model": "ollama:gemma3:12b", "axis": "governance", "bytes_match_pin": true, "id_ok": true, "sig_ok": true, "n": 237, "accuracy": 0.3629, "k_implied": 86, "wilson95": "[0.3043, 0.4258]", "parse_errors_excluded": 0, "comparison_citable": true}
{"file": "signed-conforma-02f63fcd0876.json", "model": "ollama:gemma3:12b", "axis": "conformance", "bytes_match_pin": true, "id_ok": true, "sig_ok": true, "n": 35, "accuracy": 0.6571, "k_implied": 23, "wilson95": "[0.4915, 0.7917]", "parse_errors_excluded": 0, "comparison_citable": true}
{"file": "signed-conforma-539a7bb52711.json", "model": "ollama:qwen2.5:3b", "axis": "conformance", "bytes_match_pin": true, "id_ok": true, "sig_ok": true, "n": 35, "accuracy": 0.5143, "k_implied": 18, "wilson95": "[0.3557, 0.6701]", "parse_errors_excluded": 0, "comparison_citable": true}
{"file": "signed-care-e974e29df4cb.json", "model": "ollama:qwen2.5:0.5b", "axis": "care", "bytes_match_pin": true, "id_ok": true, "si

In [4]:
assert all(r['bytes_match_pin'] and r['id_ok'] and r['sig_ok'] for r in rows), 'verification failed'
print(f'{len(rows)}/{len(rows)} cards: bytes match pin, id recomputes, Ed25519 valid under board-attestation-1')
print('Not comparison-citable (record only):', [r['file'] for r in rows if not r['comparison_citable']])

4/4 cards: bytes match pin, id recomputes, Ed25519 valid under board-attestation-1
Not comparison-citable (record only): ['signed-care-e974e29df4cb.json']


Wilson intervals use k = round(accuracy × n), because the card body publishes accuracy to 4 decimals, not k. Check k against the published rows before quoting. These cards are not yet in a card root (latest root 2026-10-12 predates G2).